In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
DATA_FILE = "uber.csv"

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully!")

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

Dataset loaded successfully!

First 5 rows:


,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5



Dataset Shape:
(200000, 9)

Column Names:
['Unnamed: 0', 'key', 'fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']


In [3]:
# Remove unnecessary spaces from column names
df.columns = [c.strip() for c in df.columns]

# Identify target column
target = next(
    (c for c in df.columns if c.lower() in ["fare_amount", "fare", "price"]),
    None
)

if target is None:
    raise ValueError("Fare target column not found!")

# Convert fare into numeric format
df[target] = pd.to_numeric(df[target], errors="coerce")

# Remove missing fare values
df = df.dropna(subset=[target])

# Remove invalid and extreme fare values
df = df[
    (df[target] > 0) &
    (df[target] < df[target].quantile(0.99))
]

print("Target column:", target)
print("Dataset shape after cleaning:", df.shape)

Target column: fare_amount
Dataset shape after cleaning: (197974, 9)


In [4]:
# Validate longitude
for c in ["pickup_longitude", "dropoff_longitude"]:
    if c in df:
        df = df[df[c].between(-180, 180)]

# Validate latitude
for c in ["pickup_latitude", "dropoff_latitude"]:
    if c in df:
        df = df[df[c].between(-90, 90)]

# Remove unnecessary key column
if "key" in df:
    df = df.drop(columns=["key"])

print("Invalid coordinates removed!")
print("Remaining dataset shape:", df.shape)

Invalid coordinates removed!
Remaining dataset shape: (197961, 8)


In [5]:
if "pickup_datetime" in df:

    dt = pd.to_datetime(
        df["pickup_datetime"],
        errors="coerce",
        utc=True
    )

    # Extract useful information from datetime
    df["pickup_hour"] = dt.dt.hour
    df["pickup_day"] = dt.dt.day
    df["pickup_month"] = dt.dt.month

    # Remove original datetime column
    df = df.drop(columns=["pickup_datetime"])

print("Feature engineering completed!")

display(df.head())

Feature engineering completed!


,Unnamed: 0,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,pickup_day,pickup_month
0,24238194,7.5,-73.999817,40.738354,-73.999512,40.723217,1,19,7,5
1,27835199,7.7,-73.994355,40.728225,-73.994710,40.750325,1,20,17,7
2,44984355,12.9,-74.005043,40.740770,-73.962565,40.772647,1,21,24,8
3,25894730,5.3,-73.976124,40.790844,-73.965316,40.803349,3,8,26,6
4,17610152,16.0,-73.925023,40.744085,-73.973082,40.761247,5,17,28,8


In [6]:
# Replace infinite values with missing values
df = df.replace([np.inf, -np.inf], np.nan)

# Separate input features and target
X = df.drop(columns=[target])
y = df[target]

print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=np.number).columns.tolist()

cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("\nNumerical Columns:", num_cols)
print("\nCategorical Columns:", cat_cols)

Input Features Shape: (197961, 9)
Target Shape: (197961,)

Numerical Columns: ['Unnamed: 0', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'pickup_hour', 'pickup_day', 'pickup_month']

Categorical Columns: []


In [7]:
# Numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
pre = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (158368, 9)
Testing Data: (39593, 9)


In [9]:
lr_pipeline = Pipeline([
    ("preprocess", pre),
    ("model", LinearRegression())
])

# Train model
lr_pipeline.fit(X_train, y_train)

# Predictions
lr_pred = lr_pipeline.predict(X_test)

# Evaluation
lr_r2 = r2_score(y_test, lr_pred)

lr_rmse = np.sqrt(
    mean_squared_error(y_test, lr_pred)
)

print("Linear Regression Results")
print("-------------------------")
print("R2 Score:", round(lr_r2, 4))
print("RMSE:", round(lr_rmse, 4))

Linear Regression Results
-------------------------
R2 Score: 0.0013
RMSE: 7.929


In [10]:
rf_pipeline = Pipeline([
    ("preprocess", pre),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train model
rf_pipeline.fit(X_train, y_train)

# Predictions
rf_pred = rf_pipeline.predict(X_test)

# Evaluation
rf_r2 = r2_score(y_test, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_pred)
)

print("Random Forest Results")
print("----------------------")
print("R2 Score:", round(rf_r2, 4))
print("RMSE:", round(rf_rmse, 4))

Random Forest Results
----------------------
R2 Score: 0.7972
RMSE: 3.5735


In [11]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "R2 Score": [lr_r2, rf_r2],
    "RMSE": [lr_rmse, rf_rmse]
})

display(results)

,Model,R2 Score,RMSE
0,Linear Regression,0.001346,7.928956
1,Random Forest,0.797158,3.573451


In [12]:
correlation = df.select_dtypes(
    include=np.number
).corr()[target].sort_values(ascending=False)

print("Correlation with Fare Amount:")
display(correlation)

Correlation with Fare Amount:


fare_amount          1.000000
pickup_month         0.026455
passenger_count      0.013403
pickup_longitude     0.004912
dropoff_longitude    0.004698
pickup_day           0.003117
Unnamed: 0          -0.000326
pickup_latitude     -0.004969
dropoff_latitude    -0.005141
pickup_hour         -0.020679
Name: fare_amount, dtype: float64